# Запускать 1 раз за использование данного блокнота

### Загрузка необходимых библиотек

In [ ]:
!pip install -U qwen_agent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.6 MB/s eta 0:00:00


## Создание агента

### Дополнительное устранение проблем

In [1]:
!pip install uv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 49.1 MB/s eta 0:00:00


In [2]:
!curl -LsSf https://astral.sh/uv/install.sh | BINDIR=/usr/local/bin bash


downloading uv 0.11.15 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [3]:
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!uv --version


uv 0.11.15 (x86_64-unknown-linux-gnu)


# Запускать каждый раз:

### Загрузка vllm для сервера
Необходимо загрузить и перезапустить среду, чтобы изменения были приняты (после перезпуска заново не запускать!)

In [4]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.9 MB/s

### Авторизируемся в HuggingFace (обязательно), чтобы не было проблем с сервером

In [4]:
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Запуск локального сервера

Запускаем сервер

In [1]:
import os
os.environ["VLLM_USE_V1"] = "0"

In [5]:
import requests
import subprocess
import time
with open("vllm_server.log", "w") as log_file:
    proc = subprocess.Popen([
        'python3', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', 'Qwen/Qwen3-4B-Instruct-2507',
        '--gpu-memory-utilization', '0.4',
        '--dtype', 'float16',
        '--max-model-len', '2048',
        '--port', '8000'
    ], stdout=log_file, stderr=log_file)

Ждем, когда запустится сервер

Нужно дождаться (2-3 минуты) строки: INFO:    Uvicorn running on http://0.0.0 (Press CTRL+C to quit)

In [1]:
!cat vllm_server.log

cat: vllm_server.log: No such file or directory


### Устранение ошибки Google Collab (fileno)

In [42]:
try:
    from ipykernel.iostream import OutStream
    OutStream.fileno = lambda self: 1
except:
    pass

### Проверка доступности сервера

In [43]:
!curl http://localhost:8000/v1/models

curl: (7) Failed to connect to localhost port 8000 after 0 ms: Connection refused


### Создание MCP-файла конфигурации

In [ ]:
from qwen_agent.agents import Assistant

# Define LLM
llm_cfg = {
    'model': 'Qwen/Qwen3-4B-Instruct-2507',

    # Use a custom endpoint compatible with OpenAI API:
    'model_server': 'http://localhost:8000/v1',
    'api_key': 'EMPTY',
}

# Define Tools
tools = [
    {'mcpServers': {  # You can specify the MCP configuration file
            'time': {
                'command': 'uvx',
                'args': ['mcp-server-time', '--local-timezone=Europe/Moscow']
            },
            "fetch": {
                "command": "uvx",
                "args": ["mcp-server-fetch"]
            }
        }
    }
]

### Создание агента

Задаём системный промпт для агента

In [ ]:
sys_mes = f"""\Ты полезный ассистент.
                 Для любого вопроса, требующего текущей даты, времени,
                 или года, всегда используй инструмент времени, чтобы получить
                 точную дату. Не полагайся на свои знания текущей даты"""

Создаём агента

In [ ]:
bot = Assistant(llm=llm_cfg, function_list=tools, system_message=sys_mes)

2026-05-20 05:55:53,093 - mcp_manager.py - 142 - INFO - Initializing MCP tools from mcp servers: ['time', 'fetch']
INFO:qwen_agent_logger:Initializing MCP tools from mcp servers: ['time', 'fetch']
2026-05-20 05:55:53,103 - mcp_manager.py - 372 - INFO - Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: time
INFO:qwen_agent_logger:Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: time
2026-05-20 05:55:55,144 - mcp_manager.py - 372 - INFO - Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: fetch
INFO:qwen_agent_logger:Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: fetch


Делаем запрос

In [ ]:
messages = [{'role': 'user', 'content': 'Сколько времени и лет прошло с 1945 года'}]
for responses in bot.run(messages=messages):
    pass

2026-05-20 05:55:57,568 - base.py - 780 - INFO - ALL tokens: 15, Available tokens: 57935
INFO:qwen_agent_logger:ALL tokens: 15, Available tokens: 57935
2026-05-20 05:56:02,172 - base.py - 780 - INFO - ALL tokens: 141, Available tokens: 57935
INFO:qwen_agent_logger:ALL tokens: 141, Available tokens: 57935


Ответ модели

In [ ]:
final_answer = [msg['content'] for msg in responses
                if msg['role'] == 'assistant' and msg.get('content')][-1]
print(final_answer)

Текущее время в Москве — 20 мая 2026 года, 08:56.

Теперь рассчитаем, сколько лет прошло с 1945 года:

2026 — 1945 = **81 год**.

Таким образом, с 1945 года прошло **81 год**.
